# Reproducible Quantum Diffusion Literature Search

This notebook is an analysis interface for the packaged pipeline. It does not duplicate acquisition, normalization, deduplication, scoring, or reporting logic.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if (ROOT / 'src').exists():
    sys.path.insert(0, str(ROOT / 'src'))

from quantum_diffusion_search.config import load_config

cfg = load_config(ROOT / 'configs/search_config.yaml')
cfg['study']

{'title': 'Quantum Diffusion Models literature search',
 'description': 'Reproducible search strategy for quantum diffusion model literature across arXiv and Crossref DOI-prefix scoped IEEE/Springer records.',
 'strategy_version': '1.0.0'}

## Search Configuration

The search strategy is defined in `configs/search_config.yaml`, including query IDs, source-specific syntax, date range, DOI prefixes, rate limits, scoring weights, and export settings.

In [2]:
pd.DataFrame(cfg['queries'])[['query_id', 'concept', 'strategy', 'arxiv_query', 'crossref_query']]

,query_id,concept,strategy,arxiv_query,crossref_query
0,Q01,quantum diffusion model,high_precision,"all:""quantum diffusion model""","""quantum diffusion model"""
1,Q02,quantum diffusion models,high_precision,"all:""quantum diffusion models""","""quantum diffusion models"""
2,Q03,quantum denoising diffusion,high_precision,"all:""quantum denoising diffusion""","""quantum denoising diffusion"""
3,Q04,quantum score-based,high_precision,"all:""quantum score-based""","""quantum score-based"""
4,Q05,score-based quantum,high_precision,"all:""score-based quantum""","""score-based quantum"""
5,Q06,stochastic Schrödinger diffusion,high_precision,"all:""stochastic Schrödinger diffusion""","""stochastic Schrödinger diffusion"""
6,Q07,stochastic Schrodinger diffusion,high_precision,"all:""stochastic Schrodinger diffusion""","""stochastic Schrodinger diffusion"""
7,Q08,quantum generative model diffusion,high_recall,"all:""quantum generative model"" AND all:""diffus...","""quantum generative model"" diffusion"
8,Q09,quantum generative models diffusion,high_recall,"all:""quantum generative models"" AND all:""diffu...","""quantum generative models"" diffusion"
9,Q10,quantum circuit diffusion model,high_recall,"all:""quantum circuit"" AND all:""diffusion model""","""quantum circuit"" ""diffusion model"""


## Load Existing Run Outputs

Run the pipeline first with `python -m quantum_diffusion_search all --config configs/search_config.yaml` or `make smoke-test`.

In [3]:
processed = ROOT / 'data' / 'processed'
all_path = processed / 'all_source_records.csv'
dedup_path = processed / 'deduplicated_records.csv'
if all_path.exists() and dedup_path.exists():
    all_records = pd.read_csv(all_path)
    deduped = pd.read_csv(dedup_path)
else:
    all_records = pd.DataFrame()
    deduped = pd.DataFrame()
len(all_records), len(deduped)

(7408, 4892)

## Counts by Source

In [4]:
all_records['database_scope'].value_counts().rename_axis('database_scope').reset_index(name='records') if not all_records.empty else pd.DataFrame(columns=['database_scope', 'records'])

,database_scope,records
0,IEEE,3600
1,Springer,3598
2,arXiv,147
3,Legacy arXiv notebook,63


## Topic and Year Distributions

In [5]:
if not deduped.empty:
    display(deduped['year'].dropna().astype(str).value_counts().sort_index().rename_axis('year').reset_index(name='records'))
    display(deduped['topic_class'].fillna('missing').value_counts().rename_axis('topic_class').reset_index(name='records'))
else:
    display(pd.DataFrame(columns=['year', 'records']))

,year,records
0,2020,241
1,2021,371
2,2022,469
3,2023,693
4,2024,1078
5,2025,1372
6,2026,668


,topic_class,records
0,missing,4840
1,quantum circuit / QNN generative model,16
2,DDPM / denoising diffusion,15
3,other / needs manual check,9
4,score-based diffusion,6
5,quantum generative model,5
6,Schrödinger bridge,1


## Duplicate Groups

In [6]:
dup_path = processed / 'duplicate_groups.csv'
pd.read_csv(dup_path).head(20) if dup_path.exists() else pd.DataFrame()

,duplicate_group_id,record_id,kept_record_id,database_scope,doi_normalized,title_normalized
0,D00001,2026-07-09T184559Z_780688a_R000001,2026-07-09T184559Z_780688a_R007367,arXiv,NaN,an hybrid quantum classical diffusion model fo...
1,D00001,2026-07-09T184559Z_780688a_R000030,2026-07-09T184559Z_780688a_R007367,arXiv,NaN,an hybrid quantum classical diffusion model fo...
2,D00001,2026-07-09T184559Z_780688a_R000059,2026-07-09T184559Z_780688a_R007367,arXiv,NaN,an hybrid quantum classical diffusion model fo...
3,D00001,2026-07-09T184559Z_780688a_R000133,2026-07-09T184559Z_780688a_R007367,arXiv,NaN,an hybrid quantum classical diffusion model fo...
4,D00001,2026-07-09T184559Z_780688a_R007367,2026-07-09T184559Z_780688a_R007367,Legacy arXiv notebook,NaN,an hybrid quantum classical diffusion model fo...
5,D00002,2026-07-09T184559Z_780688a_R000002,2026-07-09T184559Z_780688a_R007408,arXiv,NaN,effective color dipole approach to color trans...
6,D00002,2026-07-09T184559Z_780688a_R000031,2026-07-09T184559Z_780688a_R007408,arXiv,NaN,effective color dipole approach to color trans...
7,D00002,2026-07-09T184559Z_780688a_R007408,2026-07-09T184559Z_780688a_R007408,Legacy arXiv notebook,NaN,effective color dipole approach to color trans...
8,D00003,2026-07-09T184559Z_780688a_R000003,2026-07-09T184559Z_780688a_R007409,arXiv,NaN,deuteron normalization and channel dependent f...
9,D00003,2026-07-09T184559Z_780688a_R000032,2026-07-09T184559Z_780688a_R007409,arXiv,NaN,deuteron normalization and channel dependent f...


## Relevant Candidates

The relevance score is only a prioritization aid. Human screening determines final inclusion.

In [7]:
threshold = cfg['relevance']['threshold']
cols = ['relevance_score', 'year', 'topic_class', 'title', 'authors', 'doi_normalized', 'database_scope', 'landing_page_url']
if not deduped.empty:
    deduped[deduped['relevance_score'].fillna(0).astype(int) >= threshold][cols].sort_values(['relevance_score', 'year'], ascending=[False, False]).head(30)
else:
    pd.DataFrame(columns=cols)

## Limitations

The IEEE and Springer scopes use Crossref DOI-prefix metadata and are not direct searches of IEEE Xplore or Springer Nature proprietary APIs. Crossref abstracts may be missing. Metadata can change over time. Final inclusion requires human screening.